# Multimodal RAG - Upsert & Query on Qdrant

3 collections: `chunks_fixed_size`, `chunks_paragraph`, `chunks_semantic`

- Text chunks → `bkai-foundation-models/vietnamese-bi-encoder` (768d, `text_vector`)
- Image chunks → `sentence-transformers/clip-ViT-B-32` (512d, `image_vector`) — supports Vietnamese

## Text Upsert

In [1]:
# ── 2. Imports & Config ─────────────────────────────────────────────────────
import csv
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import (
    VectorParams,
    Distance,
    PointStruct,
    NamedVector,
)
from tqdm.auto import tqdm
import pandas as pd
import sentence_transformers

import torch
import numpy as np
from PIL import Image
from sentence_transformers import SentenceTransformer

# ── Config ──────────────────────────────────────────────────────────────────
QDRANT_URL  = "http://localhost:6333"
COLLECTIONS = {
    "fixed_size": "chunks_fixed_size",
    "paragraph":  "chunks_paragraph",
    "semantic":   "chunks_semantic",
}
CHUNK_CSV = {
    "fixed_size": "D:/RAG-DB/chunking_scripts/chunks_fixed_size.csv",
    "paragraph":  "D:/RAG-DB/chunking_scripts/chunks_paragraph.csv",
    "semantic":   "D:/RAG-DB/chunking_scripts/chunks_semantic.csv",
}

# Embedding models
BKVEC_MODEL   = "bkai-foundation-models/vietnamese-bi-encoder"   # text → 768d
IMG_MODEL      = "sentence-transformers/clip-ViT-B-32"  # image → 512d
TEXT_VEC_DIM  = 768
IMG_VEC_DIM   = 512

c:\Users\Admin\miniconda3\envs\vector_db\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── 3. Connect to Qdrant ────────────────────────────────────────────────────
client = QdrantClient(url=QDRANT_URL)
info = client.info()
print(f"Qdrant {info.version} | collections: {[c.name for c in client.get_collections().collections]}")

Qdrant 1.17.1 | collections: ['chunks_fixed_size', 'chunks_paragraph', 'chunks_semantic']


In [3]:
# ── 4. Create collections ───────────────────────────────────────────────────
for name, coll in COLLECTIONS.items():
    if client.collection_exists(coll):
        print(f"[SKIP] '{coll}' already exists")
        continue
    client.create_collection(
        collection_name=coll,
        vectors_config={
            "text_vector":  VectorParams(size=TEXT_VEC_DIM, distance=Distance.COSINE),  # 768d bkai
            "image_vector": VectorParams(size=IMG_VEC_DIM,  distance=Distance.COSINE),  # 768d CLIP-ViT
        },
    )
    print(f"[CREATED] '{coll}' — text_vector({TEXT_VEC_DIM}d) + image_vector({IMG_VEC_DIM}d)")
print("\nDone.")

[SKIP] 'chunks_fixed_size' already exists
[SKIP] 'chunks_paragraph' already exists
[SKIP] 'chunks_semantic' already exists

Done.


In [4]:
import pandas as pd

chunks_by_coll = {}
for strat, csv_path in CHUNK_CSV.items():
    # Đọc toàn bộ CSV vào DataFrame
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    
    # Lọc data cực nhanh bằng vectorization của Pandas
    # Sau đó chuyển ngược về dạng list of dicts (.to_dict('records')) 
    # để tương thích với hàm upsert() của bạn
    texts = df[df["modality"] == "text"].to_dict('records')
    # imgs  = df[df["modality"] == "image"].to_dict('records')
    
    chunks_by_coll[strat] = {"text": texts}
    print(f"{COLLECTIONS[strat]}: {len(texts)} text")

chunks_fixed_size: 23708 text
chunks_paragraph: 28617 text
chunks_semantic: 53409 text


In [10]:


# Xác định thiết bị (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Batch sizes
TEXT_BATCH = 16   # bkai: 768d
IMG_BATCH  = 8    # CLIP ViT-B/32: 512d

# =====================================================================
# 1. BKAi (Text -> 768d)
# =====================================================================
print("Loading bkai...")
# Khởi tạo thẳng bằng SentenceTransformer (Gọn gàng, không cần AutoModel/Tokenizer)
bkai_model = SentenceTransformer(BKVEC_MODEL, device=device)

def embed_text_bkai(texts: list[str]) -> list[list[float]]:
    """Embed text hiện đại: Giao toàn bộ việc chia batch, padding, L2-norm cho thư viện."""
    if not texts:
        return []
        
    embeddings = bkai_model.encode(
        texts,
        batch_size=TEXT_BATCH,
        normalize_embeddings=True, # Tự động làm L2-normalization
        convert_to_numpy=True,
        show_progress_bar=False
    )
    return embeddings.tolist()



Device: cuda
Loading bkai...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5403.70it/s]


In [6]:
# ── 7. Upsert (embed first, then batch-upload) ──────────────────────────────
import uuid


BATCH = 256

def _build_payload(row):
    return {
        "chunk_id":     row["chunk_id"],
        "modality":     row["modality"],
        "row_id":       int(row["row_id"]),
        "chunk_index":  int(row["chunk_index"]),
        "text":         row["text"],
        "image_path":   row["image_path"],
        "text_chunk_id": row.get("text_chunk_id", ""),
        "source":       row["source"],
        "url":          row["url"],
        "title":        row["title"],
        "author":       row["author"],
        "date":         row["date"],
        "corpus_id":    int(row["corpus_id"]),
    }

def upsert(strategy: str):
    coll = COLLECTIONS[strategy]
    rows = chunks_by_coll[strategy]

    # --- text chunks (bkai → 768d) -----------------------------------------
    text_rows = rows["text"]
    if text_rows:
        print(f"[{coll}] Embedding {len(text_rows)} text chunks with bkai...")
        text_embs = embed_text_bkai([r["text"] for r in text_rows])

        text_points = [
            PointStruct(
                id=str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{r['row_id']}_{r['chunk_index']}")),
                vector={"text_vector": emb},   # only the active vector field
                payload=_build_payload(r),
            )
            for r, emb in zip(text_rows, text_embs)
        ]
        print(f"[{coll}] Upserting {len(text_points)} text points...")
        # for i in range(0, len(text_points), BATCH):
        #     client.upsert(collection_name=coll, points=text_points[i:i+BATCH])
        client.upload_points(collection_name=coll, points=text_points, batch_size=BATCH)
        print(f"[{coll}] Text upsert done.")

    # --- image chunks (CLIP-ViT → 512d) -------------------------------------
    # img_rows = rows["image"]
    # if not img_rows:
    #     print(f"[{coll}] No image chunks. Skipping.")
    #     return

    # print(f"[{coll}] Embedding {len(img_rows)} image chunks with CLIP-ViT...")
    # img_paths = [r["image_path"] for r in img_rows]
    # img_embs  = embed_image_clip(img_paths)

    # img_points = [
    #     PointStruct(
    #         id=str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{r['row_id']}_{r['chunk_index']}")),
    #         vector={"image_vector": emb},   # only the active vector field
    #         payload=_build_payload(r),
    #     )
    #     for r, emb in zip(img_rows, img_embs)
    # ]
    # print(f"[{coll}] Upserting {len(img_points)} image points...")
    # for i in range(0, len(img_points), BATCH):
    #     client.upsert(collection_name=coll, points=img_points[i:i+BATCH])
    # print(f"[{coll}] Image upsert done.")


# for strat in COLLECTIONS:
#     # if (strat == "semantic"):
#     print(f"\n{'='*50}")
#     upsert(strat)
# print("\n✓ All collections upserted.")

## Image Upsert

In [3]:
corpus = pd.read_csv("./FinalDataset/final_corpus.csv", encoding="utf-8-sig")

In [4]:
# 2. Viết hàm tách chuỗi media thành list các đường dẫn
def split_media(media_str):
    if pd.isna(media_str):
        return []
    return {p.strip() for p in str(media_str).split(' | ') if p.strip()}

# Tạo cột mới 'image_path' chứa list các đường dẫn
corpus['image_path'] = corpus['media'].apply(split_media)

# 3. Explode DataFrame
# Bước này sẽ bung list trong 'image_path' thành nhiều dòng
# Các dòng không có ảnh (NaN) sẽ bị loại bỏ
df_exploded = corpus.explode('image_path').dropna(subset=['image_path'])

print(f"Tổng số ảnh trích xuất được (bao gồm có thể trùng lặp/lỗi): {len(df_exploded)}")

# 4. Lấy danh sách các đường dẫn UNIQUE để kiểm tra (tiết kiệm thời gian check I/O)
unique_paths = df_exploded['image_path'].unique()

Tổng số ảnh trích xuất được (bao gồm có thể trùng lặp/lỗi): 9085


In [5]:
from concurrent.futures import ThreadPoolExecutor
import os

def is_valid_image(path):
    path = os.path.join("./FinalDataset", path)
    if not os.path.exists(path):
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False

# 5. Chạy kiểm tra song song
print(f"Đang kiểm tra {len(unique_paths)} đường dẫn duy nhất...")
with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(executor.map(is_valid_image, unique_paths))

# Tạo một tập hợp (set) chứa các đường dẫn HỢP LỆ để tra cứu siêu tốc
valid_paths_set = set([path for path, is_valid in zip(unique_paths, results) if is_valid])

# 6. Lọc lại DataFrame, chỉ giữ những dòng có image_path nằm trong danh sách hợp lệ
df_valid = df_exploded[df_exploded['image_path'].isin(valid_paths_set)].copy()

print(f"Số lượng payload hợp lệ có thể tạo: {len(df_valid)}")


Đang kiểm tra 8880 đường dẫn duy nhất...
Số lượng payload hợp lệ có thể tạo: 9059


In [6]:
# =====================================================================
# 2. CLIP-ViT (Image -> 512d)
# =====================================================================
print("Loading CLIP-ViT...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_model = SentenceTransformer(IMG_MODEL, device=device)
dimension = img_model.get_sentence_embedding_dimension()
print(f"The embedding dimension is: {dimension}")



def load_valid_image(path):
    """Giữ nguyên logic cực kỳ cẩn thận này của bạn để check file hỏng."""
    path = os.path.join("./FinalDataset", path)
    if not (os.path.exists(path) and os.path.isfile(path)):
        return None
    try:
        with Image.open(path) as img:
            img.verify() 
        return Image.open(path)
    except Exception:
        return None

def embed_image_clip(image_paths: list[str]) -> list[list[float]]:
    """Embed image hiện đại: Xử lý an toàn các file hỏng để không làm lệch index khi zip() upsert."""
    if not image_paths:
        return []
    
    # print(image_paths)
    # return []
    # Khởi tạo trước một ma trận toàn số 0.
    # Nếu ảnh hỏng, nó sẽ giữ nguyên vector [0, 0, ..., 0]
    final_embs = np.zeros((len(image_paths), dimension), dtype=np.float32)
    
    valid_images = []
    valid_indices = []
    
    # Chỉ load những ảnh hợp lệ và lưu lại vị trí (index) gốc của nó
    for idx, path in enumerate(image_paths):
        img = load_valid_image(path)
        if img is not None:
            valid_images.append(img)
            valid_indices.append(idx)
        else:
            print(f"[WARN] Invalid image skipped: {path}")
            
    # Tiến hành embed toàn bộ ảnh hợp lệ một lần
    if valid_images:
        valid_embs = img_model.encode(
            valid_images,
            batch_size=8,
            normalize_embeddings=True, # Tự động làm L2-normalization
            convert_to_numpy=True,
            show_progress_bar=False
        )
        
        # Lắp ráp các vector hợp lệ về đúng vị trí index gốc của chúng
        for i, original_idx in enumerate(valid_indices):
            final_embs[original_idx] = valid_embs[i]
            
    return final_embs.tolist()

def search_image(image_path: str, collection: str, top_k: int = 5):
    """Search image_vector (multilingual-CLIP 512d) by image file."""
    emb = embed_image_clip([image_path])[0]
    results = client.query_points(
        collection_name=collection,
        query=emb,
        using="image_vector",
        limit=top_k,
        with_payload=True,
        with_vectors=False,
    )
    return results.points

Loading CLIP-ViT...


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 7250.12it/s]
CLIPModel LOAD REPORT from: sentence-transformers/clip-ViT-B-32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The embedding dimension is: 512


C:\Users\Admin\AppData\Local\Temp\ipykernel_7224\1686549242.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = img_model.get_sentence_embedding_dimension()


In [7]:
 # --- image chunks (CLIP-ViT → 512d) -------------------------------------
# img_rows = rows["image"]
# if not img_rows:
#     print(f"[{coll}] No image chunks. Skipping.")
#     return
from ast import MatchValue
import gc
import uuid
import math

from qdrant_client.grpc import FieldCondition
from qdrant_client.models import Filter


def _build_payload_image(row):
    return {
        "modality":     "image",
        "corpus_id":       int(row["id"]),
        "image_path":   row["image_path"],
        "source":       row["source"],
        "url":          row["url"],
        "title":        row["title"],
        "author":       row["author"],
        "date":         row["date"],
    }

def upsert_images(collection: str, df_valid: pd.DataFrame, batch_size: int = 128):
    total_images = len(df_valid)
    num_batches = math.ceil(total_images / batch_size)
    
    print(f"[{collection}] Bắt đầu xử lý {total_images} ảnh, chia thành {num_batches} batches (kích thước {batch_size})...")
    
    # Bọc vòng lặp bằng tqdm
    for i in tqdm(range(num_batches), desc=f"[{collection}] Upserting", unit="batch"):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, total_images)
        
        # 1. Cắt lấy 1 batch
        df_batch = df_valid.iloc[start_idx:end_idx]
        
        # 2. Embed
        try:
            batch_embs = embed_image_clip(df_batch["image_path"].tolist())
        except Exception as e:
            # Dùng tqdm.write thay vì print để không làm trôi/vỡ thanh tiến trình
            tqdm.write(f"\n[{collection}] Lỗi khi embed batch {i+1}: {e}")
            continue
            
        assert len(batch_embs) == len(df_batch), f"Lệch số lượng vector ở batch {i+1}!"

        # 3. Build PointStruct
        batch_points = [
            PointStruct(
                id=str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{r['id']}_{r['image_path']}")),
                vector={"image_vector": emb},
                payload=_build_payload_image(r),
            )
            for r, emb in zip(df_batch.to_dict(orient="records"), batch_embs)
        ]
        
        # 4. Upload
        client.upload_points(collection_name=collection, points=batch_points)
        
        # 5. Dọn dẹp RAM
        del df_batch, batch_embs, batch_points
        gc.collect()

    print(f"[{collection}] Hoàn tất Image upsert.")
    
    
from qdrant_client import models


for strat in COLLECTIONS:
    coll = COLLECTIONS[strat]
    client.delete(
        collection_name=coll,
        points_selector=Filter(
            must=[
                models.FieldCondition(
                    key="modality",
                    match=models.MatchValue(value="image"),
                ),
            ]
        )
    )
    print(f"[{coll}] Đã xóa tất cả điểm có modality='image'. Bắt đầu upsert lại...")
    print(f"\n{'='*50}")
    upsert_images(coll, df_valid, batch_size=512)

[chunks_fixed_size] Đã xóa tất cả điểm có modality='image'. Bắt đầu upsert lại...

[chunks_fixed_size] Bắt đầu xử lý 9059 ảnh, chia thành 18 batches (kích thước 512)...


[chunks_fixed_size] Upserting: 100%|██████████| 18/18 [04:21<00:00, 14.53s/batch]


[chunks_fixed_size] Hoàn tất Image upsert.
[chunks_paragraph] Đã xóa tất cả điểm có modality='image'. Bắt đầu upsert lại...

[chunks_paragraph] Bắt đầu xử lý 9059 ảnh, chia thành 18 batches (kích thước 512)...


[chunks_paragraph] Upserting: 100%|██████████| 18/18 [05:21<00:00, 17.83s/batch]


[chunks_paragraph] Hoàn tất Image upsert.
[chunks_semantic] Đã xóa tất cả điểm có modality='image'. Bắt đầu upsert lại...

[chunks_semantic] Bắt đầu xử lý 9059 ảnh, chia thành 18 batches (kích thước 512)...


[chunks_semantic] Upserting: 100%|██████████| 18/18 [04:39<00:00, 15.55s/batch]

[chunks_semantic] Hoàn tất Image upsert.


## Testing

In [8]:
# ── 9. Collection stats ─────────────────────────────────────────────────────
for name, coll in COLLECTIONS.items():
    info = client.get_collection(coll)
    print(f"{coll}: {info.points_count} points  |  indexed on text_vector + image_vector")

chunks_fixed_size: 32767 points  |  indexed on text_vector + image_vector
chunks_paragraph: 37676 points  |  indexed on text_vector + image_vector
chunks_semantic: 62468 points  |  indexed on text_vector + image_vector


In [13]:
# ── 8. Multimodal search ────────────────────────────────────────────────────
def search_text(query: str, collection: str, top_k: int = 5):
    """Search text_vector (bkai 768d)."""
    emb = embed_text_bkai([query])[0]
    results = client.query_points(
          collection_name=collection,
          query=emb,               # the embedding vector
          using="text_vector",     # which named vector field to search
          limit=top_k,
          with_payload=True,
          with_vectors=False,
    )
    return results.points        # return the list of hits directly


# ── Example usage ────────────────────────────────────────────────────────────
COLLECTION = COLLECTIONS["semantic"]  # change to "paragraph" or "semantic"

# 1) Text query
print(f"─── Text search in '{COLLECTION}' ───")
results = search_text("Con của Trịnh Huệ sinh năm 2021 đã được làm thủ tục nhập khẩu thường trú đầy đủ.", COLLECTION, top_k=3)
for r in results:
    print(f"  score={r.score:.4f}  [{r.payload['modality']}]  {str(r.payload.get('text',''))[:80]}...")

# 2) Image query — replace with a real image path from your media folder
print(f"\n─── Image search in '{COLLECTION}' ───")
img_results = search_image("news_media/anh-man-hinh-2026-03-20-luc-163751-17739999845371561792287.png", COLLECTION, top_k=3)
for r in img_results:
    print(f"  score={r.score:.4f}  [{r.payload['modality']}]  {r.payload['image_path']}")

─── Text search in 'chunks_semantic' ───
  score=0.4206  [text]  Vậy, tôi muốn đăng ký nhập khẩu thường trú cho con tôi thì cần những thủ tục gì?...
  score=0.4071  [text]  Sau khi đăng ký thường trú về nơi ở mới tại xã Bình Mỹ, công dân thực hiện thủ t...
  score=0.3833  [text]  (sinh năm 2022) là con anh Hoàng Văn Tuệ (sinh năm 1998, trú tại xã Văn Bàn, tỉn...

─── Image search in 'chunks_semantic' ───
  score=1.0000  [image]  news_media/anh-man-hinh-2026-03-20-luc-163751-17739999845371561792287.png
  score=0.8153  [image]  news_media/photo-library-20260321123828-a29d19f4-7722-4627-9828-e30442d6f18a-z7642820187013-bbc755de4b8b05e9f21b8e68dbaec9b3.jpg
  score=0.8000  [image]  news_media/photo-library-20260315172240-87ceded4-06e3-41a5-aa35-8af356b5740b-1-muong-ly.jpg
